In [26]:
import glob
import os
import numpy as np
import xarray as xr
import geopandas as gpd
import regionmask
import warnings
import matplotlib.pyplot as plt
from scipy import ndimage

# Suppress specific xarray/dask warnings for cleaner output
warnings.filterwarnings("ignore")

# ==========================================
# Phase 1: Configuration Loader
# ==========================================
class PipelineConfig:
    """Centralized configuration manager for the pipeline."""
    def __init__(self, model_name, target_variable):
        self.model_name = model_name
        self.target_variable = target_variable

        #Hardcoding built-in paths
        self.base_path = "../CMIP6 data"
        self.shapefile_path = "../data/AmazonBasinLimits-master/amazon_sensulatissimo_gmm_v1.shp"
        
        # Default spatial bounding box for Amazon basin
        self.lat_min = -20
        self.lat_max = 10
        self.lon_min = 280
        self.lon_max = 320

    def get_dict(self):
        """Returns the configuration as a dictionary."""
        return {
            "model_name": self.model_name,
            "target_variable": self.target_variable,
            "base_path": self.base_path,
            "shapefile_path": self.shapefile_path,
            "lat_min": self.lat_min,
            "lat_max": self.lat_max,
            "lon_min": self.lon_min,
            "lon_max": self.lon_max
        }

# ==========================================
# Phase 2: Data Fetcher
# ==========================================
class DataFetcher:
    """Dynamically loads NetCDF files and handles safe time alignment."""
    def __init__(self, config):
        self.config = config
        self.model = config["model_name"]
        self.base_dir = config["base_path"]

    def _build_filepath(self, variable, experiment, frequency):
        """Constructs the file path pattern and verifies existence."""
        file_pattern = f"{variable}_{frequency}_{self.model}_{experiment}_r1i1p1f1_*.nc"
        full_path = os.path.join(self.base_dir, self.model, experiment, file_pattern)
        
        if not glob.glob(full_path):
            raise FileNotFoundError(f"No files found: {full_path}")
        return full_path

    def _safe_time_align(self, ds_target, ds_reference, target_name):
        """Validate and align monthly time coordinates between datasets."""
        if len(ds_target.time) != len(ds_reference.time):
            raise ValueError(f"Time length mismatch in {target_name}.")
            
        target_ym = ds_target.time.dt.strftime("%Y-%m").values
        ref_ym = ds_reference.time.dt.strftime("%Y-%m").values
        
        if not np.array_equal(target_ym, ref_ym):
            raise ValueError(f"Year-Month values do not match in {target_name}.")
            
        ds_target["time"] = ds_reference["time"]
        return ds_target

    def fetch_data(self):
        """Returns a dictionary of required raw datasets."""
        datasets = {}
        target_var = self.config["target_variable"]
        tas_files = glob.glob(self._build_filepath("tas", "1pctCO2", "Amon"))
        with xr.open_dataset(tas_files[0]) as ds_first:
            datasets["attrs_raw"] = ds_first.attrs.copy()

        # 1. Load temperature data for GWL
        datasets["tas_1pct"] = xr.open_mfdataset(self._build_filepath("tas", "1pctCO2", "Amon"))
        datasets["tas_pi"] = xr.open_mfdataset(self._build_filepath("tas", "piControl", "Amon"), combine="by_coords")

        # 2. Complex Variable Branch (MCWD and so on)
        if target_var == "MCWD":
            pr_ds = xr.open_mfdataset(
                self._build_filepath("pr", "1pctCO2", "Amon"),
                combine="by_coords",
            )

            evspsbl_ds = xr.open_mfdataset(
                self._build_filepath("evspsbl", "1pctCO2", "Amon"),
                combine="by_coords",
            )

            datasets["pr"] = pr_ds
            datasets["evspsbl"] = self._safe_time_align(
                evspsbl_ds,
                pr_ds,
                "evspsbl",
            )
        
        elif target_var == "AnnualRainfall":
            datasets["pr"] = xr.open_mfdataset(
              self._build_filepath("pr", "1pctCO2", "Amon")
            )
        
        # if target_var == "NEW_VARIABLE":
        #     pr_ds = xr.open_mfdataset(self._build_filepath("OTHER VARIABLE", "1pctCO2", "Amon or Lmon"))
        #     datasets["OTHER VARIABLE"] = VARIABL_NAME_ds

        # 3. Direct Variable Branch (Single Variable like cVeg)
        else:
            freq = "Amon" if target_var in ["tas", "pr"] else "Lmon"
            datasets["target_raw"] = xr.open_mfdataset(self._build_filepath(target_var, "1pctCO2", freq))

        return datasets

# ==========================================
# Phase 3: Core Processors
# ==========================================
class BaseProcessor:
    def __init__(self, config):
        self.config = config
        self.target_var = config["target_variable"]
        self.lat_slice = slice(config["lat_min"], config["lat_max"])
        self.lon_slice = slice(config["lon_min"], config["lon_max"])

    def process(self, datasets):
        raise NotImplementedError

class DirectVariableProcessor(BaseProcessor):
    """Processes simple variables by taking the annual mean."""
    def process(self, datasets):
        raw_da = datasets["target_raw"][self.target_var]
        original_attrs = raw_da.attrs.copy()
        sliced_da = datasets["target_raw"][self.target_var].sel(lat=self.lat_slice, lon=self.lon_slice)
        annual_da = sliced_da.resample(time="1YS").mean().load()
        annual_da.attrs = original_attrs
        annual_da.name = self.target_var
        return annual_da

class MCWDProcessor(BaseProcessor):
    """Calculate hydrological-year MCWD from precipitation and evapotranspiration."""

    HYDROLOGICAL_MONTHS = np.array(
        [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    )

    def process(self, datasets):
        pr = datasets["pr"]["pr"].sel(
            lat=self.lat_slice,
            lon=self.lon_slice,
        )

        evspsbl = datasets["evspsbl"]["evspsbl"].sel(
            lat=self.lat_slice,
            lon=self.lon_slice,
        )

        self._validate_inputs(pr, evspsbl)

        # Ensure that both variables use exactly the same time coordinate
        evspsbl = evspsbl.assign_coords(time=pr["time"])

        # Convert monthly mean fluxes from kg m-2 s-1 to mm month-1
        seconds_per_month = (
            pr["time"].dt.days_in_month.astype("float64") * 86400.0
        )

        monthly_precipitation = pr * seconds_per_month
        monthly_evaporation = evspsbl * seconds_per_month

        monthly_water_balance = (
            monthly_precipitation - monthly_evaporation
        ).load()

        years = monthly_water_balance["time"].dt.year.values.astype(int)
        months = monthly_water_balance["time"].dt.month.values.astype(int)

        # Oct-Dec belong to the hydrological year ending in the next year
        hydrological_years = np.where(
            months >= 10,
            years + 1,
            years,
        )

        # Create annual timestamps using the same year-start convention as GWL
        annual_time = (
            pr.isel(lat=0, lon=0)
            .resample(time="1YS")
            .mean()
            ["time"]
        )

        time_lookup = {
            int(year): timestamp
            for year, timestamp in zip(
                annual_time.dt.year.values.astype(int),
                annual_time.values,
            )
        }

        annual_results = []

        for hydrological_year in np.unique(hydrological_years):
            indices = np.where(
                hydrological_years == hydrological_year
            )[0]

            # Skip incomplete Oct-Sep years
            if len(indices) != 12:
                continue

            water_balance_year = monthly_water_balance.isel(
                time=indices
            )

            template = water_balance_year.isel(time=0, drop=True)

            cumulative_water_deficit = np.zeros_like(
                template.values,
                dtype=np.float64,
            )

            minimum_cwd = np.zeros_like(
                cumulative_water_deficit,
                dtype=np.float64,
            )

            for time_index in range(12):
                monthly_balance = water_balance_year.isel(
                    time=time_index
                ).values

                cumulative_water_deficit = np.minimum(
                    0.0,
                    cumulative_water_deficit + monthly_balance,
                )

                minimum_cwd = np.minimum(
                    minimum_cwd,
                    cumulative_water_deficit,
                )

            # Keep negative MCWD values
            mcwd_values = minimum_cwd

            hydro_year = int(hydrological_year)

            if hydro_year not in time_lookup:
                continue

            mcwd_year = xr.DataArray(
                mcwd_values,
                coords=template.coords,
                dims=template.dims,
                name="MCWD",
            ).expand_dims(
                time=[time_lookup[hydro_year]]
            )

            annual_results.append(mcwd_year)

        mcwd_annual = xr.concat(
            annual_results,
            dim="time",
        ).sortby("time")

        mcwd_annual.attrs = {
            "long_name": (
                "Maximum cumulative water deficit "
                "for the October-September hydrological year"
            ),
            "units": "mm",
            "calculation": (
                "min(CWD); CWD_i = min("
                "0, CWD_i-1 + precipitation_i - evapotranspiration_i)"
            ),
            "hydrological_year": "October to September",
        }


        # # Apply a 5-year centered rolling mean before converting time to GWL
        # mcwd_annual = (
        #     mcwd_annual
        #     .rolling(
        #         time=5,
        #         center=True,
        #         min_periods=5,
        #     )
        #     .mean()
        #     .dropna(dim="time", how="all")
        # )

        # mcwd_annual.name = "MCWD"
        # mcwd_annual.attrs = {
        #     "long_name": (
        #         "Smoothed maximum cumulative water deficit "
        #         "for the October-September hydrological year"
        #     ),
        #     "units": "mm",
        #     "calculation": (
        #         "min(CWD); CWD_i = min("
        #         "0, CWD_i-1 + precipitation_i - evapotranspiration_i)"
        #     ),
        #     "processing": "5-year centered rolling mean",
        #     "hydrological_year": "October to September",
        # }

        return mcwd_annual

    @staticmethod
    def _validate_inputs(pr, evspsbl):
        """Validate time coordinates and physical units."""

        pr_year_month = pr["time"].dt.strftime("%Y-%m").values
        et_year_month = evspsbl["time"].dt.strftime("%Y-%m").values

        if not np.array_equal(pr_year_month, et_year_month):
            raise ValueError(
                "pr and evspsbl do not have matching monthly timestamps."
            )

        pr_units = pr.attrs.get("units", "")
        et_units = evspsbl.attrs.get("units", "")

        expected_units = {
            "kg m-2 s-1",
            "kg m-2 s^-1",
            "kg m**-2 s**-1",
        }

        if pr_units not in expected_units:
            warnings.warn(
                f"Unexpected pr units: {pr_units}. "
                "Expected a water flux in kg m-2 s-1."
            )

        if et_units not in expected_units:
            warnings.warn(
                f"Unexpected evspsbl units: {et_units}. "
                "Expected a water flux in kg m-2 s-1."
            )    

class AnnualRainfallProcessor(BaseProcessor):
    """Convert monthly precipitation flux to annual rainfall totals."""

    def process(self, datasets):
        pr = datasets["pr"]["pr"].sel(
            lat=self.lat_slice,
            lon=self.lon_slice,
        )

        original_attrs = pr.attrs.copy()

        # Convert kg m-2 s-1 to monthly precipitation depth in mm
        seconds_per_month = pr["time"].dt.days_in_month * 86400
        monthly_rainfall = pr * seconds_per_month

        # Sum monthly precipitation to obtain mm per year
        annual_rainfall = (
            monthly_rainfall
            .resample(time="1YS")
            .sum()
            .load()
        )

        annual_rainfall.name = "AnnualRainfall"
        annual_rainfall.attrs = original_attrs
        annual_rainfall.attrs["units"] = "mm yr-1"
        annual_rainfall.attrs["long_name"] = (
            "Annual total precipitation"
        )

        return annual_rainfall

class ProcessorFactory:
    """Routes data to the correct processor."""

    @staticmethod
    def get_processor(config):
        target_var = config["target_variable"]

        if target_var == "MCWD":
            return MCWDProcessor(config)

        if target_var == "AnnualRainfall":
            return AnnualRainfallProcessor(config)

        return DirectVariableProcessor(config)
    
class GWLCalculator:
    """Calculate GWL following the Terpstra et al. approach."""

    @staticmethod
    def calculate_global_annual_mean(ds):
        tas = ds["tas"]

        weights = np.cos(np.deg2rad(ds["lat"]))
        weights.name = "weights"

        global_monthly_tas = (
            tas
            .weighted(weights)
            .mean(dim=["lat", "lon"], skipna=True)
        )

        global_annual_tas = (
            global_monthly_tas
            .resample(time="1YS")
            .mean(dim="time")
            .load()
        )

        return global_annual_tas

    @staticmethod
    def compute_gwl(tas_1pct_ds, tas_pi_ds):
        # Annual global mean temperatures
        gmt_1pct = GWLCalculator.calculate_global_annual_mean(
            tas_1pct_ds
        )

        gmt_pi = GWLCalculator.calculate_global_annual_mean(
            tas_pi_ds
        )

        # Terpstra et al.: mean over the complete piControl simulation
        gmt_preindustrial = float(
            gmt_pi.mean(dim="time").values
        )

        # Terpstra et al.: Gaussian smoothing with sigma = 10 years
        gmt_1pct_smoothed_values = ndimage.gaussian_filter1d(
            gmt_1pct.values,
            sigma=10,
            mode="nearest",
        )

        gmt_1pct_smoothed = xr.DataArray(
            gmt_1pct_smoothed_values,
            dims=["time"],
            coords={"time": gmt_1pct["time"]},
            name="GMT_smoothed",
        )

        # GWL relative to the piControl climatological mean
        gwl = gmt_1pct_smoothed - gmt_preindustrial

        gwl.name = "GWL"
        gwl.attrs["units"] = "K"
        gwl.attrs["baseline"] = (
            "Mean GMT over the complete piControl simulation"
        )
        gwl.attrs["smoothing"] = (
            "Gaussian filter with sigma=10 years, mode='nearest'"
        )

        print(f"piControl baseline: {gmt_preindustrial:.4f} K")
        print(
            "GWL range:",
            f"{float(gwl.min()):.3f} to "
            f"{float(gwl.max()):.3f} K",
        )

        return gwl

# ==========================================
# Phase 4: Spatial Masker
# ==========================================
class SpatialMasker:
    """Applies the Amazon basin shapefile mask."""
    def __init__(self, config):
        self.shapefile_path = config["shapefile_path"]
        
    def apply_mask(self, data_array):
        amazon_boundary = gpd.read_file(self.shapefile_path)
        if amazon_boundary.crs is not None and amazon_boundary.crs.to_string() != "EPSG:4326":
            amazon_boundary = amazon_boundary.to_crs(epsg=4326)
        amazon_single_basin = amazon_boundary.unary_union
        
        lon_original = data_array["lon"]
        lon_shifted = data_array["lon"].where(data_array["lon"] <= 180, data_array["lon"] - 360)
        da_shifted = data_array.assign_coords(lon=lon_shifted)
        
        basin_region = regionmask.Regions([amazon_single_basin])
        mask = basin_region.mask(da_shifted["lon"], da_shifted["lat"], wrap_lon=False)
        
        masked_da_shifted = da_shifted.where(mask == 0)
        masked_da = masked_da_shifted.assign_coords(lon=lon_original)
        
        return masked_da

# ==========================================
# Phase 5: TOAD Formatter & Exporter
# ==========================================
class TOADExporter:
    """Formats and exports to a TOAD-compatible NetCDF file."""
    def __init__(self, config):
        self.model = config["model_name"]
        self.variable = config["target_variable"]

    def export(self, masked_da, gwl_da, raw_attrs):
        merged_ds = xr.merge([masked_da, gwl_da], join="inner")
        final_ds = merged_ds.set_coords("GWL").swap_dims({"time": "GWL"}).sortby("GWL")
        
        final_ds.attrs = raw_attrs
        final_ds.attrs["nominal_resolution"] = raw_attrs.get("nominal_resolution", "Not Available")

        vars_to_drop = [var for var in final_ds.data_vars if var != self.variable]
        final_ds = final_ds.drop_vars(vars_to_drop)
        
        output_dir = "./processed_data"
        os.makedirs(output_dir, exist_ok=True)
        output_filename = f"TOAD_{self.variable}_{self.model}.nc"
        output_path = os.path.join(output_dir, output_filename)
        
        final_ds.to_netcdf(output_path)
        print(f"[Export Success] Created NetCDF at: {output_path}")
        return output_path


# ==========================================
# Function 1: Generate and Save .nc file
# ==========================================
def generate_nc(model_name, target_variable):
    """Executes Phase 1 to 5 to process raw CMIP6 data and save it as a .nc file."""
    print(f"\n--- Generating .nc for {model_name} | {target_variable} ---")
    
    # Paths are now automatically handled inside PipelineConfig
    config_manager = PipelineConfig(model_name, target_variable)
    config = config_manager.get_dict()
    
    fetcher = DataFetcher(config)
    datasets = fetcher.fetch_data()
    
    processor = ProcessorFactory.get_processor(config)
    target_da = processor.process(datasets)
    gwl_da = GWLCalculator.compute_gwl(datasets["tas_1pct"], datasets["tas_pi"])
    
    masker = SpatialMasker(config)
    masked_target_da = masker.apply_mask(target_da)
    
    print("Target time size:", target_da.sizes.get("time", 0))
    print("GWL time size:", gwl_da.sizes.get("time", 0))

    print("Target first times:")
    print(target_da["time"].values[:3])

    print("GWL first times:")
    print(gwl_da["time"].values[:3])

    common_times = np.intersect1d(
        target_da["time"].values,
        gwl_da["time"].values,
    )

    print("Common time size:", len(common_times))

    exporter = TOADExporter(config)
    output_nc_path = exporter.export(masked_target_da, gwl_da, datasets["attrs_raw"])
    
    return output_nc_path

# ==========================================
# Function 2: Load .nc file and Plot
# ==========================================
def plot_trend_from_nc(model_name, target_variable):
    """Loads a previously processed .nc file and generates a spatial mean plot."""
    file_path = f"./processed_data/TOAD_{target_variable}_{model_name}.nc"
    if not os.path.exists(file_path):
        print(f"[Error] File not found: {file_path}")
        return
        
    print(f"\n--- Plotting data from {file_path} ---")
    ds = xr.open_dataset(file_path)
    
    # Calculate Spatial Mean using lon and lat
    spatial_mean = ds[target_variable].mean(dim=["lon", "lat"], skipna=True)
        
    plot_metadata = {
        "cVeg": {"color": "forestgreen", "ylabel": "Mean Vegetation Carbon ($kgC/m^2$)"},
        "MCWD": {"color": "darkred", "ylabel": "Mean Max Cumulative Water Deficit (mm)"},
        "cSoil": {"color": "sienna", "ylabel": "Mean Soil Carbon ($kgC/m^2$)"},
        "treeFrac": {"color": "darkgreen", "ylabel": "Mean Tree Cover (%)"},
        "grassFrac": {"color": "limegreen", "ylabel": "Mean Grass Area (%)"},
        "baresoilFrac": {"color": "tan", "ylabel": "Mean Bare Soil Cover (%)"},
        "fVegLitter": {"color": "olive", "ylabel": "Mean Veg to Litter Flux ($kgC/m^2/s$)"},
        "fFire": {"color": "firebrick", "ylabel": "Mean Fire CO2 Emission Flux ($kgC/m^2/s$)"},
        "gpp": {"color": "teal", "ylabel": "Mean Gross Primary Production ($kgC/m^2/s$)"},
        "lai": {"color": "mediumseagreen", "ylabel": "Mean Leaf Area Index ($m^2/m^2$)"},
        "rGrowth": {"color": "purple", "ylabel": "Mean Autotrophic Respiration ($kgC/m^2/s$)"}
    }
    
    meta = plot_metadata.get(target_variable, {"color": "blue", "ylabel": f"Mean {target_variable}"})
    gwl_vals = spatial_mean["GWL"].values
    var_vals = spatial_mean.values
    
    plt.figure(figsize=(8, 5))
    plt.plot(gwl_vals, var_vals, color=meta["color"], linewidth=2.5, label=f"{model_name} {target_variable}")
    
    plt.title(f"Amazon Area-Averaged {target_variable} Response to GWL ({model_name})", fontsize=13)
    plt.xlabel("Global Warming Level (GWL) [°C]", fontsize=11)
    plt.ylabel(meta["ylabel"], fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()

    output_dir = "./plots"
    os.makedirs(output_dir, exist_ok=True)
    output_filename = f"TrendPlot_{target_variable}_{model_name}.png"
    output_path = os.path.join(output_dir, output_filename)
    
    plt.savefig(output_path, dpi=300)
    plt.close()
    
    print(f"[Plot Success] Saved graph at: {output_path}")

# ==========================================
# Function 3: Plot GWL graph for model
# ==========================================
def plot_gwl_by_model_year(model_name):
    """Calculate and plot GWL against 1pctCO2 model year."""

    config = PipelineConfig(
        model_name=model_name,
        target_variable="tas",
    ).get_dict()

    fetcher = DataFetcher(config)

    tas_1pct = xr.open_mfdataset(
        fetcher._build_filepath(
            "tas",
            "1pctCO2",
            "Amon",
        ),
        combine="by_coords",
    )

    tas_pi = xr.open_mfdataset(
        fetcher._build_filepath(
            "tas",
            "piControl",
            "Amon",
        ),
        combine="by_coords",
    )

    gwl = GWLCalculator.compute_gwl(
        tas_1pct,
        tas_pi,
    )

    model_years = np.arange(1, gwl.sizes["time"] + 1)

    plt.figure(figsize=(8, 5))

    plt.plot(
        model_years,
        gwl.values,
        linewidth=2.5,
        label=model_name,
    )

    plt.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1,
    )

    plt.title(
        f"Global Warming Level by Model Year ({model_name})",
        fontsize=13,
    )

    plt.xlabel("1pctCO2 Model Year", fontsize=11)
    plt.ylabel("Global Warming Level (GWL) [°C]", fontsize=11)

    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()

    output_dir = "./plots"
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(
        output_dir,
        f"GWL_ModelYear_{model_name}.png",
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()

    tas_1pct.close()
    tas_pi.close()

    print(f"[Plot Success] Saved GWL graph at: {output_path}")


# ==========================================
# Function 4: Load .nc file and Plot Spatial Maps
# ==========================================
def plot_spatial_maps_from_nc(model_name, target_variable):
    """Loads a processed .nc file and generates a 1x6 grid of spatial maps."""
    file_path = f"./processed_data/TOAD_{target_variable}_{model_name}.nc"
    if not os.path.exists(file_path):
        print(f"[Error] File not found: {file_path}")
        return
        
    print(f"\n--- Plotting spatial maps from {file_path} ---")
    ds = xr.open_dataset(file_path)
    data_array = ds[target_variable]
    
    plot_metadata = {
        "cVeg": {"cmap": "YlGn", "label": "Vegetation Carbon ($kgC/m^2$)"},
        "MCWD": {"cmap": "OrRd", "label": "Max Cumulative Water Deficit (mm)"},
        "cSoil": {"cmap": "YlOrBr", "label": "Soil Carbon ($kgC/m^2$)"},
        "treeFrac": {"cmap": "Greens", "label": "Tree Cover (%)"},
        "grassFrac": {"cmap": "YlGn", "label": "Grass Area (%)"},
        "baresoilFrac": {"cmap": "Oranges", "label": "Bare Soil Cover (%)"},
        "fVegLitter": {"cmap": "YlGnBu", "label": "Veg to Litter Flux ($kgC/m^2/s$)"},
        "fFire": {"cmap": "Reds", "label": "Fire CO2 Emission ($kgC/m^2/s$)"},
        "gpp": {"cmap": "Greens", "label": "Gross Primary Production ($kgC/m^2/s$)"},
        "lai": {"cmap": "summer", "label": "Leaf Area Index ($m^2/m^2$)"},
        "rGrowth": {"cmap": "Purples", "label": "Autotrophic Respiration ($kgC/m^2/s$)"}
    }
    
    meta = plot_metadata.get(target_variable, {"cmap": "viridis", "label": f"{target_variable}"})

    target_gwls = [0.0, 1.5, 2.0, 3.0, 4.0, 5.0]
    v_min = float(data_array.min(skipna=True))
    v_max = float(data_array.max(skipna=True))

    # Built-in shapefile path
    shapefile_path = "../data/AmazonBasinLimits-master/amazon_sensulatissimo_gmm_v1.shp"
    amazon_boundary = gpd.read_file(shapefile_path)
    if amazon_boundary.crs is not None and amazon_boundary.crs.to_string() != "EPSG:4326":
        amazon_boundary = amazon_boundary.to_crs(epsg=4326)
    
    if data_array["lon"].max() > 180:
        amazon_boundary_plot = amazon_boundary.translate(xoff=360)
    else:
        amazon_boundary_plot = amazon_boundary

    fig, axes = plt.subplots(1, 6, figsize=(24, 4.5), sharex=True, sharey=True)

    for ax, target in zip(axes, target_gwls):
        da_slice = data_array.sel(GWL=target, method="nearest")
        actual_gwl = float(da_slice["GWL"])

        mesh = ax.pcolormesh(
            da_slice["lon"],
            da_slice["lat"],
            da_slice.values,
            cmap=meta["cmap"],
            vmin=v_min,
            vmax=v_max,
            shading="auto",
        )

        amazon_boundary_plot.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.5)
        ax.set_title(f"GWL: {target}°C\n(Actual: {actual_gwl:.2f}°C)", fontsize=11)
        ax.set_xlabel("Longitude [°E]", fontsize=9)

    axes[0].set_ylabel("Latitude [°N]", fontsize=10)

    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(mesh, cax=cbar_ax)
    cbar.set_label(meta["label"], fontsize=11)

    plt.subplots_adjust(right=0.90)
    plt.suptitle(f"Amazon ${target_variable}$ Spatial Distribution by Global Warming Level ({model_name})", fontsize=14, y=1.05)
    
    output_dir = "./plots"
    os.makedirs(output_dir, exist_ok=True)
    output_filename = f"SpatialMap_{target_variable}_{model_name}.png"
    output_path = os.path.join(output_dir, output_filename)
    
    plt.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close()
    
    print(f"[Plot Success] Saved spatial map at: {output_path}")

### Attention : Path for CMIP6 Model output data


<pre>
CMIP6 data/
├── GFDL-ESM4/
│   ├── 1pctCO2/
│   │   └── tas_Amon_GFDL-ESM4_1pctCO2_r1i1p1f1_*.nc ... 
│   ├── piControl/
│   │   └── cVeg_Lmon_GFDL-ESM4_esm-piControl_r1i1p1f1_gr1_*.nc ...
│   └── SSP/
├── MPI-ESM1-2-LR/
│   ├── 1pctCO2/
│   └── piControl/
├── NorCPM1/
│   ├── 1pctCO2/
│   └── piControl/
└── TaiESM1/
    ├── 1pctCO2/
    └── piControl/
</pre>

- Case Sensitivity: Ensure model folder names (e.g., GFDL-ESM4) and experiment names (e.g., 1pctCO2) match exactly as listed above. (Download the folders directly from the Google drive as it saved)

- File Pattern: The pipeline uses a wildcard pattern("*") to load data. Ensure your .nc files are placed directly inside the respective experiment folder (e.g., 1pctCO2/).

- Path Verification: If you encounter a FileNotFoundError, the error message will display the specific path the code attempted to access. Compare that path with your actual local file directory.

### EXECUTE SESSION : Please use the functions here for preprocessing!

In [91]:
# Target Configurations
MODEL = "GFDL-ESM4"
VARIABLE = "treeFrac"

In [32]:
# 1. Plot model year - GWL graph
plot_gwl_by_model_year(MODEL)

piControl baseline: 286.6624 K
GWL range: 0.103 to 5.080 K
[Plot Success] Saved GWL graph at: ./plots/GWL_ModelYear_MPI-ESM1-2-LR.png


In [92]:
# 2. Processing and saving .nc files
generate_nc(MODEL, VARIABLE)

# 3. Loading saved .nc file and plotting 2D Trend/Spatial Maps
plot_trend_from_nc(MODEL, VARIABLE)
plot_spatial_maps_from_nc(MODEL, VARIABLE)


--- Generating .nc for GFDL-ESM4 | treeFrac ---
piControl baseline: 286.5637 K
GWL range: -0.019 to 3.992 K
Target time size: 150
GWL time size: 150
Target first times:
[cftime.DatetimeNoLeap(1, 1, 1, 0, 0, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2, 1, 1, 0, 0, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(3, 1, 1, 0, 0, 0, 0, has_year_zero=True)]
GWL first times:
[cftime.DatetimeNoLeap(1, 1, 1, 0, 0, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2, 1, 1, 0, 0, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(3, 1, 1, 0, 0, 0, 0, has_year_zero=True)]
Common time size: 150
[Export Success] Created NetCDF at: ./processed_data/TOAD_treeFrac_GFDL-ESM4.nc

--- Plotting data from ./processed_data/TOAD_treeFrac_GFDL-ESM4.nc ---
[Plot Success] Saved graph at: ./plots/TrendPlot_treeFrac_GFDL-ESM4.png

--- Plotting spatial maps from ./processed_data/TOAD_treeFrac_GFDL-ESM4.nc ---
[Plot Success] Saved spatial map at: ./plots/SpatialMap_treeFrac_GFDL-ESM4.png
